# LangGraph and MCP: A Revision Notebook
### State graphs, reducers, checkpointing, tool calling, ReAct, HITL, FastMCP servers, and multi-server clients

**Goal:** understand the graph model that powers agent workflows, then connect graphs to external tools through the Model Context Protocol.

**Covered:**

1. StateGraph
2. State
3. Reducers
4. Nodes
5. Edges
6. START / END
7. Memory / Checkpointer
8. Tool calling
9. ReAct agent
10. HITL
11. FastMCP server - stdio
12. FastMCP server - HTTP
13. MCP client
14. Multi-server MCP client


## 1. Basic LangGraph

A graph is state flowing through nodes connected by edges. This section builds the smallest useful graph: one node between `START` and `END`.


In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


### 1.1 State

State is the data that flows through the graph.

```text
      State
        ↓
    ┌───────┐
    │ graph │
    └───────┘
        ↓
   updated State
```

A node maps `State → State`.


In [ ]:
class State(TypedDict):
    message: str


### 1.2 Node

A node is simply a function that:

1. receives state,
2. does something,
3. returns state updates.


In [ ]:
def hello_node(state: State):
    return {
        "message": "Hello " + state["message"]
    }


### 1.3 Build the graph

Create the graph and add the node.


In [ ]:
builder = StateGraph(State)


builder.add_node(
    "hello",
    hello_node
)


### 1.4 Edges

```text
START
  ↓
hello
  ↓
END
```


In [ ]:
builder.add_edge(
    START,
    "hello"
)

builder.add_edge(
    "hello",
    END
)


### 1.5 Compile


In [ ]:
graph = builder.compile()


### 1.6 Invoke


In [ ]:
result = graph.invoke(
    {
        "message": "Pratyush"
    }
)

print(result)


## 2. Multiple nodes

Chain `double` and `add_ten` so state flows through both nodes.


In [ ]:
class MathState(TypedDict):
    number: int
    result: int


def double_node(state: MathState):
    return {
        "result": state["number"] * 2
    }


def add_ten_node(state: MathState):
    return {
        "result": state["result"] + 10
    }


In [ ]:
builder = StateGraph(MathState)

builder.add_node("double", double_node)
builder.add_node("add_ten", add_ten_node)

builder.add_edge(START, "double")
builder.add_edge("double", "add_ten")
builder.add_edge("add_ten", END)

graph = builder.compile()


In [ ]:
result = graph.invoke(
    {
        "number": 5,
        "result": 0
    }
)

print(result)


Execution order:

```text
State
  ↓
double
  ↓
add_ten
  ↓
END
```

State changes:

```text
number=5
result=0
   ↓
double
result=10
   ↓
add_ten
result=20
```


## 3. Reducers

Without a reducer, a node's returned value may replace the old state value.

With `Annotated[list, add]`, the reducer says: **old value + new value**.

| Step | Value |
|---|---|
| Old | `["hello"]` |
| Node returns | `["world"]` |
| Reducer | `add` |
| Result | `["hello", "world"]` |


In [ ]:
from typing import Annotated
from operator import add


class MessageState(TypedDict):
    messages: Annotated[list[str], add]


In [ ]:
def node_one(state: MessageState):
    return {
        "messages": ["hello"]
    }


def node_two(state: MessageState):
    return {
        "messages": ["world"]
    }


In [ ]:
builder = StateGraph(MessageState)

builder.add_node("one", node_one)
builder.add_node("two", node_two)

builder.add_edge(START, "one")
builder.add_edge("one", "two")
builder.add_edge("two", END)

graph = builder.compile()


result = graph.invoke(
    {
        "messages": []
    }
)

print(result)


## 4. Memory and checkpointing

A thread identifies one conversation.

```text
Same thread_id
      ↓
Same checkpointed state
      ↓
Graph can remember previous state
```

A different thread, such as `thread_id="user-456"`, gets separate memory.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver


class ChatState(TypedDict):
    messages: Annotated[list[str], add]


def chat_node(state: ChatState):
    return {
        "messages": ["AI: I remember this conversation."]
    }


builder = StateGraph(ChatState)

builder.add_node(
    "chat",
    chat_node
)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)


checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


In [ ]:
config = {
    "configurable": {
        "thread_id": "user-123"
    }
}


graph.invoke(
    {
        "messages": ["Human: My name is Pratyush."]
    },
    config=config
)


Later, invoke the same graph with the same config:


In [ ]:
graph.invoke(
    {
        "messages": ["Human: What is my name?"]
    },
    config=config
)


## 5. Tool calling in LangGraph

Typical LangGraph tool architecture:

```text
                 ┌─────────────┐
                 │    LLM      │
                 └──────┬──────┘
                        │
                   tool call?
                   /         \
                 YES          NO
                  │            │
                  ▼            ▼
                TOOL          END
                  │
                  ▼
              Tool result
                  │
                  └──────────→ LLM
```

The graph controls this loop.


In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str):
    """Get the weather for a city."""
    return f"It is sunny in {city}."


In modern LangGraph you can use `ToolNode`.


In [ ]:
from langgraph.prebuilt import ToolNode


tools = [get_weather]

tool_node = ToolNode(tools)


The `ToolNode`:

```text
receives AIMessage containing tool_calls
        ↓
executes appropriate tools
        ↓
returns ToolMessage(s)
```


## 6. ReAct agent

ReAct is:

```text
REASON → ACT → OBSERVE → REASON → ACT → ...
```

Example:

```text
User:
    "What's the weather?"

LLM:
    I need weather information.

Action:
    get_weather("Kolkata")

Observation:
    Sunny

LLM:
    Final answer.
```


In [ ]:
from langgraph.prebuilt import create_react_agent


> **Before you run:** this section expects a `model` object. Define one as in Chapter 1, for example with `ChatGoogleGenerativeAI` or `ChatOllama`.


In [ ]:
react_agent = create_react_agent(
    model,
    tools=[get_weather]
)


result = react_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Kolkata?"
            }
        ]
    }
)


print(result)


`create_react_agent` is a prebuilt implementation of the ReAct-style agent loop.

Instead of manually constructing:

```text
LLM node
  ↓
conditional edge
  ↓
ToolNode
  ↓
LLM
```

you can use the prebuilt agent.


## 7. Human-in-the-loop

HITL = **Human In The Loop**. It is used when the AI wants to perform an action that requires human approval.

```text
User
  ↓
LLM
  ↓
"I want to send this email."
  ↓
INTERRUPT
  ↓
HUMAN
  ↓
approve / edit / reject
  ↓
continue
```


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


Modern LangGraph HITL can use an interrupt.


In [ ]:
from langgraph.types import interrupt


def send_email_node(state):
    approval = interrupt(
        {
            "question": "Send this email?",
            "email": state["email"]
        }
    )

    if approval == "yes":
        return {
            "status": "Email sent"
        }

    return {
        "status": "Email cancelled"
    }


Human resumes the graph:


In [ ]:
graph.invoke(
    Command(
        resume="yes"
    ),
    config=config
)


### HITL three core actions

| Action | What happens |
|---|---|
| Approve | Execute exactly the proposed action |
| Edit | Modify the proposed action, then execute the modified version |
| Reject | Do not execute the tool/action |

```text
1. APPROVE
   AI proposed send_email(...)
   HUMAN approves
   → execute exactly that action.

2. EDIT
   AI proposed send_email(to="wrong@email.com")
   HUMAN edits
   → send_email(to="correct@email.com")
   → modified action executes.

3. REJECT
   AI proposed delete_file(...)
   HUMAN rejects
   → tool/action does NOT execute.
```

Memory and checkpointing are important because the graph needs to pause and later resume from exactly where it stopped.


## 8. MCP

MCP = Model Context Protocol.

The important idea: MCP standardizes how an AI application discovers and interacts with external tools, resources, and prompts.

Instead of:

```text
AI application
     ↓
custom API integration
```

MCP gives:

```text
AI application
     ↓
    MCP
     ↓
MCP server
     ↓
 tools/data
```

- **MCP server:** exposes tools, resources, and prompts.
- **MCP client:** connects to an MCP server.
- **AI application:** uses the discovered capabilities.


## 9. FastMCP server: stdio

File: `weather_server.py`


In [ ]:
from fastmcp import FastMCP


mcp = FastMCP("Weather Server")


@mcp.tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It is sunny in {city}."


if __name__ == "__main__":
    mcp.run(
        transport="stdio"
    )


STDIO architecture:

```text
MCP CLIENT
    │
    │ stdin/stdout
    ▼
FastMCP Server
    │
    ▼
get_weather()
```

There is no HTTP server. The client launches the MCP server as a subprocess and communicates through standard input/output.


## 10. FastMCP server: HTTP

The same server can expose HTTP transport.


In [ ]:
mcp = FastMCP("Weather HTTP Server")


@mcp.tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It is sunny in {city}."


if __name__ == "__main__":
    mcp.run(
        transport="http",
        host="127.0.0.1",
        port=8000
    )


HTTP architecture:

```text
MCP CLIENT
    │
    │ HTTP
    ▼
http://127.0.0.1:8000
    │
    ▼
FastMCP Server
    │
    ▼
get_weather()
```

| Transport | Communication |
|---|---|
| STDIO | Local process communication |
| HTTP | Network communication |

The MCP protocol remains the same idea; the transport is different.


## 11. MCP client

A client connects to an MCP server and discovers its tools.

```text
MCP Client
    │
    │ connect
    ▼
MCP Server
    │
    ▼
list_tools()
    │
    ▼
get_weather
```

The client does not need to hard-code every tool's implementation. It can discover capabilities exposed by the server.


In [ ]:
from fastmcp import Client


async def main():

    async with Client(
        "weather_server.py"
    ) as client:

        tools = await client.list_tools()

        print(tools)

        result = await client.call_tool(
            "get_weather",
            {
                "city": "Kolkata"
            }
        )

        print(result)


Client endpoints:

```python
# STDIO client:
Client("weather_server.py")

# HTTP client:
Client("http://127.0.0.1:8000/mcp")
```


## 12. Multi-server MCP client

This is where MCP becomes particularly useful. Your application can connect to multiple MCP servers.

```text
                   AI APPLICATION
                         │
                   MCP CLIENT
                         │
            ┌────────────┼────────────┐
            ▼            ▼            ▼
       Weather MCP    GitHub MCP   Database MCP
         Server         Server        Server
            │            │            │
       weather()      search()      query()
```

The application gets tools from multiple independent servers.


In [ ]:
servers = {
    "weather": {
        "transport": "stdio",
        "command": "python",
        "args": ["weather_server.py"],
    },

    "github": {
        "transport": "http",
        "url": "http://localhost:9000/mcp",
    },

    "database": {
        "transport": "http",
        "url": "http://localhost:8000/mcp",
    },
}


The client connects to each server, discovers its tools, and aggregates them for the application.

Resulting tool set:

```text
get_weather()
search_github()
query_database()
```

The LLM can then choose among tools from multiple MCP servers.


## 13. LangGraph + MCP

This is the important final architecture.

```text
                   USER
                    │
                    ▼
                LANGGRAPH
                    │
                    ▼
                   LLM
                    │
              "I need a tool"
                    │
                    ▼
               MCP CLIENT
                    │
         ┌──────────┼──────────┐
         ▼          ▼          ▼
      Weather     GitHub    Database
       MCP         MCP        MCP
      Server      Server     Server
         │          │          │
         └──────────┼──────────┘
                    │
                Tool result
                    │
                    ▼
                   LLM
                    │
                    ▼
                  USER
```

**LangGraph manages:**

- state
- graph execution
- memory
- tool loop
- branching
- HITL

**MCP manages:**

- standardized external tool integration
- tool discovery
- server/client communication
- multiple external tool servers


## 14. The big picture

```text
             ┌──────────────────┐
             │      USER        │
             └────────┬─────────┘
                      │
                      ▼
             ┌──────────────────┐
             │    LANGGRAPH     │
             │                  │
             │     State        │
             │     Nodes        │
             │     Edges        │
             │     Memory       │
             │     HITL         │
             └────────┬─────────┘
                      │
                      ▼
             ┌──────────────────┐
             │       LLM        │
             └────────┬─────────┘
                      │
                tool call
                      │
                      ▼
             ┌──────────────────┐
             │    MCP CLIENT    │
             └────────┬─────────┘
                      │
         ┌────────────┼────────────┐
         ▼            ▼            ▼
      MCP Server   MCP Server   MCP Server
      Weather      GitHub       Database
         │            │            │
         └────────────┼────────────┘
                      │
                   result
                      │
                      ▼
                     LLM
                      │
                      ▼
                     USER
```


## 15. Revision cheat sheet

| Term | Meaning |
|---|---|
| `StateGraph(State)` | Define the graph state |
| `add_node()` | Add computation to the graph |
| `add_edge()` | Define the execution path |
| `START` | Graph entry point |
| `END` | Graph exit point |
| `STATE` | Shared data flowing through the graph |
| `REDUCER` | Tells LangGraph how to combine old + new state values |
| `CHECKPOINTER` | Saves graph state |
| `thread_id` | Identifies a conversation/execution thread |
| `TOOL NODE` | Executes tools requested by the LLM |
| `REACT` | Reason → act → observe → repeat |
| `HITL` | Pause the graph for a human decision: approve, edit, or reject |
| `MCP SERVER` | Exposes tools/resources/prompts |
| `FASTMCP` | Easy Python framework for creating MCP servers |
| `STDIO` | Local process communication |
| `HTTP` | Network communication |
| `MCP CLIENT` | Connects to an MCP server and discovers/calls capabilities |
| `MULTI-SERVER MCP` | One application/client connects to many MCP servers |


## 16. The difference between LangGraph and MCP

| Question | LangGraph | MCP |
|---|---|---|
| Core question | "How does my AI application think and execute?" | "How does my AI application connect to external capabilities?" |
| Main concepts | State, nodes, edges, loops, memory, HITL, agent execution | Tools, resources, prompts, servers, clients, standardized communication |

Therefore:

```text
LangGraph ≠ MCP
```

They solve different problems.

- LangGraph can use MCP tools.
- MCP does not replace LangGraph's state machine.


## Final memory trick

| Layer | Chain |
|---|---|
| LangGraph | graph → state → nodes → edges → loop → tools → memory → HITL |
| MCP | client → server → tools → stdio / HTTP |
| Multi-server MCP | client → multiple servers |
| Final architecture | user → LangGraph → LLM → MCP client → MCP servers → result → LLM → user |


## Self-test

1. What flows through a LangGraph graph?
2. What does a reducer do, and what happens without one?
3. What is the difference between using the same `thread_id` and a new one?
4. What does `ToolNode` do?
5. What are the three HITL actions?
6. What does an MCP server expose?
7. How do stdio and HTTP transports differ?
8. How do LangGraph and MCP complement each other?


## Official references

- [LangGraph documentation](https://docs.langchain.com/oss/python/langgraph/overview)
- [Model Context Protocol](https://modelcontextprotocol.io)
- [FastMCP documentation](https://gofastmcp.com)
